In [1]:
# this code takes raw network files and cleans it for use in the student movement model

In [2]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [3]:
edges = gpd.read_file("./network_files/walk_edges.geojson")
nodes = gpd.read_file("./network_files/walk_nodes.geojson")

In [4]:
edges = edges.to_crs(2240)
nodes = nodes.to_crs(2240)

In [5]:
# setting node ids
nodes["node_id"] = nodes["OBJECTID"] 

# if there are missing values, default to 0, which means a walkable path that isnt a building entrance
nodes.loc[nodes["kind"].isna(), "kind"] = 0 

In [6]:
# categories converting to int
nodes["kind"] = nodes["kind"].astype(int)

In [7]:
# De-duplicate co-located nodes to avoid `sjoin_nearest` returning ties
# (only drops duplicates when all nodes at the same coordinate share the same `kind`)
nodes = nodes.copy()

nodes["_x"] = nodes.geometry.x
nodes["_y"] = nodes.geometry.y
nodes["_xy_key"] = nodes["_x"].round(6).astype(str) + "_" + nodes["_y"].round(6).astype(str)

kind_nunique = nodes.groupby("_xy_key")["kind"].nunique(dropna=False)
dedup_keys = kind_nunique[kind_nunique == 1].index  # safe to collapse

if (nodes["_xy_key"].duplicated().any() and len(dedup_keys)):
    keep_dedup = (
        nodes[nodes["_xy_key"].isin(dedup_keys)]
        .sort_values("node_id")
        .groupby("_xy_key", as_index=False)
        .head(1)
    )
    keep_others = nodes[~nodes["_xy_key"].isin(dedup_keys)]
    dropped = len(nodes) - (len(keep_dedup) + len(keep_others))
    nodes = gpd.GeoDataFrame(
        pd.concat([keep_others, keep_dedup], ignore_index=True),
        geometry="geometry",
        crs=nodes.crs,
    )
    print(f"Deduped co-located nodes: dropped {dropped} duplicate rows")

nodes = nodes.drop(columns=["_x", "_y", "_xy_key"])


Deduped co-located nodes: dropped 1 duplicate rows


In [8]:
# resetting index and renaming to edge_oid
edges = edges.reset_index(drop=False).rename(columns={"index":"edge_oid"})

In [9]:
def start_point(linestring):
    coords = list(linestring.coords)
    return Point(coords[0])

def end_point(linestring):
    coords = list(linestring.coords)
    return Point(coords[-1])

In [10]:
# Build START and END points for each edge

edge_start_pts = gpd.GeoDataFrame(
    {"edge_oid": edges["edge_oid"]},
    geometry=edges.geometry.apply(start_point), crs=edges.crs
)

edge_end_pts = gpd.GeoDataFrame(
    {"edge_oid": edges["edge_oid"]},
    geometry=edges.geometry.apply(end_point), crs=edges.crs
)

In [11]:
# Nearest-join start/end points to nodes (within 1 meter)

start_join = gpd.sjoin_nearest(
    edge_start_pts, nodes[["node_id","geometry"]],
    how="left", max_distance=1.0, distance_col="dist"
).rename(columns={"node_id":"from_id"}).drop(columns=["index_right"])

end_join = gpd.sjoin_nearest(
    edge_end_pts, nodes[["node_id","geometry"]],
    how="left", max_distance=1.0, distance_col="dist"
).rename(columns={"node_id":"to_id"}).drop(columns=["index_right"])

In [12]:
# warn if any endpoints failed to match a node within 1 m

n_missing_from = start_join["from_id"].isna().sum()
n_missing_to   = end_join["to_id"].isna().sum()

if n_missing_from or n_missing_to:
    print(f"WARNING: {n_missing_from} start endpoints and {n_missing_to} end endpoints did not find a node within 1 m.")

    print("Increase max_distance (e.g., 1.5–2.0) or run a snap/integrate step to tighten endpoints.")


In [13]:
# Join those IDs back onto the original edges

edges2 = edges.merge(
    start_join[["edge_oid","from_id"]], on="edge_oid", how="left"
).merge(
    end_join[["edge_oid","to_id"]], on="edge_oid", how="left"
)

In [14]:
edges2["from_id"] = edges2["from_id"].astype("Int64")  # nullable int
edges2["to_id"]   = edges2["to_id"].astype("Int64")

In [15]:
if "length_ft" not in edges2.columns:
    # length in CRS units (assumes meters); if feet, convert
    edges2["length_ft"] = edges2.length

if "time_s" not in edges2.columns:
    v_walk = 4.5  # ft/s for walkspeed
    edges2["time_s"] = edges2["length_ft"] / v_walk

In [16]:
edges2.to_file("./network_files/walk_edges_clean.geojson", driver="GeoJSON")
nodes.to_file("./network_files/walk_nodes_clean.geojson", driver="GeoJSON")

In [17]:
m = edges2.explore()
nodes.explore(m=m, color='red')

m